In [1]:
# %%
# Импорты и настройки

import gc
import math
from pathlib import Path

import numpy as np
import polars as pl

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score


HORIZON = 24.0

TO_DROP = [
    "ид_события",
    "ид_канала_данных",
    "дата",
    "время",
    "datetime",
    "hours_since_prev_target",
]

TARGET_COLS = [
    "target_lower",
    "target_upper",
]


# Чанки можно задавать отдельно
TRAIN_CHUNKS = [*range(4)]
VAL_CHUNKS = [*range(20)]
TEST_CHUNKS = [*range(20)]


DROP_PER_STEP = 10
MIN_FEATURES = 10

# Сколько строк validation использовать для SHAP
SHAP_ROWS = 50_000


OUTPUT_DIR = Path("feature_selection_shap_rfe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_PATH = OUTPUT_DIR / "history.csv"


MODEL_PARAMS = {
    "loss_function": "Logloss",
    "eval_metric": "PRAUC:type=Classic",
    "iterations": 100,
    "depth": 6,
    "learning_rate": 0.05,
    "random_seed": 42,
    "verbose": 1,
}

In [2]:
# %%
# LazyFrame для нужного split

def scan_split(
    split: str,
    chunks: list[int],
) -> pl.LazyFrame:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .drop(TO_DROP)
        .with_columns(
            (
                (pl.col("target_upper") != -1)
                & (pl.col("target_lower") <= HORIZON)
            )
            .cast(pl.Int8)
            .alias("target")
        )
    )


# Схему берем без загрузки данных
train_schema = scan_split(
    "train",
    [TRAIN_CHUNKS[0]],
).collect_schema()


feature_cols = [
    c
    for c in train_schema.names()
    if c not in TARGET_COLS + ["target"]
]


cat_features = [
    c
    for c in feature_cols
    if train_schema[c] in (
        pl.String,
        pl.Categorical,
        pl.Enum,
    )
]


print(f"Features:  {len(feature_cols)}")
print(f"Cat feats: {len(cat_features)}")

Features:  378
Cat feats: 4


In [3]:
# %%
# Служебные функции

def collect_df(
    split: str,
    chunks: list[int],
    features: list[str],
    cat_features_current: list[str],
    with_target: bool = True,
    head: int | None = None,
) -> pl.DataFrame:

    cols = features + (["target"] if with_target else [])

    lf = (
        scan_split(split, chunks)
        .select(cols)
    )

    # Null в категориях заполняем до collect
    if cat_features_current:
        lf = lf.with_columns(
            pl.col(cat_features_current).fill_null("null")
        )

    if head is not None:
        lf = lf.head(head)

    return lf.collect(
        engine="streaming"
    )


def make_pool(
    split: str,
    chunks: list[int],
    features: list[str],
    cat_features_current: list[str],
    return_y: bool = False,
) -> Pool | tuple[Pool, np.ndarray]:

    df = collect_df(
        split=split,
        chunks=chunks,
        features=features,
        cat_features_current=cat_features_current,
        with_target=True,
    )

    y = df["target"].to_numpy()

    pool = Pool(
        data=df.select(features),
        label=y,
        cat_features=cat_features_current,
    )

    del df
    gc.collect()

    if return_y:
        return pool, y

    del y
    gc.collect()

    return pool

In [4]:
# %%
# Test score считаем по чанкам
# В RAM одновременно лежит только один test chunk

def get_test_pr_auc(
    model: CatBoostClassifier,
    features: list[str],
    cat_features_current: list[str],
) -> float:

    y_parts = []
    p_parts = []

    for chunk in TEST_CHUNKS:

        test_pool, y_chunk = make_pool(
            split="test",
            chunks=[chunk],
            features=features,
            cat_features_current=cat_features_current,
            return_y=True,
        )

        p_chunk = model.predict_proba(
            test_pool
        )[:, 1]

        y_parts.append(y_chunk)
        p_parts.append(p_chunk)

        del test_pool
        del y_chunk
        del p_chunk

        gc.collect()


    # Для точного PR-AUC нужны prediction по всему test
    # Это только два одномерных массива, поэтому памяти мало
    y_test = np.concatenate(y_parts)
    p_test = np.concatenate(p_parts)

    score = average_precision_score(
        y_test,
        p_test,
    )

    del y_parts
    del p_parts
    del y_test
    del p_test

    gc.collect()

    return score

In [5]:
# %%
# SHAP считаем частями по validation chunks
# Полная SHAP-матрица для всего validation в RAM не создается

def get_mean_abs_shap(
    model: CatBoostClassifier,
    features: list[str],
    cat_features_current: list[str],
) -> np.ndarray:

    shap_sum = np.zeros(
        len(features),
        dtype=np.float64,
    )

    total_rows = 0

    rows_per_chunk = math.ceil(
        SHAP_ROWS / len(VAL_CHUNKS)
    )


    for chunk in VAL_CHUNKS:

        shap_df = collect_df(
            split="val",
            chunks=[chunk],
            features=features,
            cat_features_current=cat_features_current,
            with_target=False,
            head=rows_per_chunk,
        )

        if shap_df.height == 0:
            del shap_df
            gc.collect()
            continue


        shap_pool = Pool(
            data=shap_df,
            cat_features=cat_features_current,
        )

        del shap_df
        gc.collect()


        shap_values = model.get_feature_importance(
            shap_pool,
            type="ShapValues",
        )


        shap_sum += np.abs(
            shap_values[:, :-1]
        ).sum(axis=0)

        total_rows += shap_values.shape[0]


        del shap_pool
        del shap_values

        gc.collect()


    if total_rows == 0:
        raise ValueError("Не удалось получить строки для SHAP")


    return shap_sum / total_rows

In [6]:
# %%
# Feature selection
# На каждом шаге:
# 1. Загружаем только текущие фичи
# 2. Обучаем модель
# 3. Сразу удаляем train/val Pool
# 4. Считаем test по чанкам
# 5. Считаем SHAP на ограниченной части validation
# 6. Убираем 10 худших фичей

history = []

current_features = feature_cols.copy()

removed_this_step = []

step = 0


while True:

    print()
    print("=" * 80)
    print(
        f"step={step} | "
        f"features={len(current_features)}"
    )


    current_cat_features = [
        c
        for c in cat_features
        if c in current_features
    ]


    # Train Pool
    train_pool = make_pool(
        split="train",
        chunks=TRAIN_CHUNKS,
        features=current_features,
        cat_features_current=current_cat_features,
        return_y=False,
    )


    # Validation Pool
    val_pool, y_val = make_pool(
        split="val",
        chunks=VAL_CHUNKS,
        features=current_features,
        cat_features_current=current_cat_features,
        return_y=True,
    )


    # Обучение
    model = CatBoostClassifier(
        **MODEL_PARAMS
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
    )


    # Validation score
    p_val = model.predict_proba(
        val_pool
    )[:, 1]

    val_pr_auc = average_precision_score(
        y_val,
        p_val,
    )


    # Train и validation больше не нужны
    del train_pool
    del val_pool
    del y_val
    del p_val

    gc.collect()


    # Test score по одному чанку
    test_pr_auc = get_test_pr_auc(
        model=model,
        features=current_features,
        cat_features_current=current_cat_features,
    )


    print(
        f"val={val_pr_auc:.6f} | "
        f"test={test_pr_auc:.6f}"
    )


    # Сохраняем результат шага
    history.append({
        "step": step,
        "n_features": len(current_features),
        "val_pr_auc": val_pr_auc,
        "test_pr_auc": test_pr_auc,
        "best_iteration": model.get_best_iteration(),
        "removed_features": " | ".join(
            removed_this_step
        ),
    })


    pl.DataFrame(
        history
    ).write_csv(
        HISTORY_PATH
    )


    # Если дошли до минимального числа фичей
    if len(current_features) <= MIN_FEATURES:

        del model

        gc.collect()

        break


    # SHAP на ограниченной части validation
    mean_abs_shap = get_mean_abs_shap(
        model=model,
        features=current_features,
        cat_features_current=current_cat_features,
    )


    shap_importance = (
        pl.DataFrame({
            "feature": current_features,
            "mean_abs_shap": mean_abs_shap,
        })
        .sort("mean_abs_shap")
    )


    n_drop = min(
        DROP_PER_STEP,
        len(current_features) - MIN_FEATURES,
    )


    removed_this_step = (
        shap_importance
        .head(n_drop)
        ["feature"]
        .to_list()
    )


    print("remove:")

    for feature in removed_this_step:
        print(f"# {feature}")


    removed_set = set(
        removed_this_step
    )


    current_features = [
        c
        for c in current_features
        if c not in removed_set
    ]


    step += 1


    del mean_abs_shap
    del shap_importance
    del model

    gc.collect()


step=0 | features=378
0:	learn: 0.2747841	test: 0.3788630	best: 0.3788630 (0)	total: 801ms	remaining: 1m 19s
1:	learn: 0.2272784	test: 0.2970486	best: 0.3788630 (0)	total: 1.65s	remaining: 1m 20s
2:	learn: 0.2522702	test: 0.2969423	best: 0.3788630 (0)	total: 2.38s	remaining: 1m 16s
3:	learn: 0.2646773	test: 0.3073097	best: 0.3788630 (0)	total: 3.15s	remaining: 1m 15s
4:	learn: 0.2685130	test: 0.3164721	best: 0.3788630 (0)	total: 3.94s	remaining: 1m 14s
5:	learn: 0.2786058	test: 0.3133867	best: 0.3788630 (0)	total: 4.66s	remaining: 1m 13s
6:	learn: 0.2961576	test: 0.3239591	best: 0.3788630 (0)	total: 5.37s	remaining: 1m 11s
7:	learn: 0.3293217	test: 0.3771652	best: 0.3788630 (0)	total: 6.17s	remaining: 1m 10s
8:	learn: 0.3807813	test: 0.3832724	best: 0.3832724 (8)	total: 7s	remaining: 1m 10s
9:	learn: 0.3960948	test: 0.3850182	best: 0.3850182 (9)	total: 7.83s	remaining: 1m 10s
10:	learn: 0.4212307	test: 0.3567374	best: 0.3850182 (9)	total: 8.66s	remaining: 1m 10s
11:	learn: 0.4231384	t

In [7]:
# %%
# Итоговая таблица

history_df = pl.read_csv(
    HISTORY_PATH
)

history_df

step,n_features,val_pr_auc,test_pr_auc,best_iteration,removed_features
i64,i64,f64,f64,i64,str
0,378,0.432087,0.708795,99,""""""
1,368,0.485435,0.541079,99,"""value_cat__Батарея неисправна …"
2,358,0.477696,0.624031,99,"""hour | value_cat__Неисправен__…"
3,348,0.475237,0.53001,91,"""value_cat__Выключен__sum_3d | …"
4,338,0.489594,0.686088,99,"""value_cat__Рычаг сдернут__sum_…"
…,…,…,…,…,…
33,48,0.464544,0.723241,97,"""value_cat__Не замкнут__sum_1h …"
34,38,0.472069,0.663278,99,"""pressure_msl (hPa)__future_12h…"
35,28,0.451483,0.583909,99,"""value_cat__Неисправен__sum_4h …"


In [15]:
history_df.sort('test_pr_auc	test_pr_auc'.split())

step,n_features,val_pr_auc,test_pr_auc,best_iteration,removed_features
i64,i64,f64,f64,i64,str
37,10,0.451438,0.502571,99,"""snow_depth (m)__future_current…"
11,268,0.466606,0.516111,99,"""value_cat__Норма__sum_2d | val…"
3,348,0.475237,0.53001,91,"""value_cat__Выключен__sum_3d | …"
1,368,0.485435,0.541079,99,"""value_cat__Батарея неисправна …"
35,28,0.451483,0.583909,99,"""value_cat__Неисправен__sum_4h …"
…,…,…,…,…,…
26,118,0.490266,0.729916,99,"""value_cat__В норме от +3 до +4…"
17,208,0.493035,0.741818,97,"""value_cat__Обнаружено движение…"
30,78,0.488799,0.743885,99,"""precipitation (mm)__future_4h …"


In [16]:
# %%
# Берем набор фичей на выбранном шаге

SELECTED_STEP = 28

history_df = pl.read_csv(HISTORY_PATH)

removed_features = (
    history_df
    .filter(pl.col("step") <= SELECTED_STEP)
    .select("removed_features")
    .drop_nulls()
    ["removed_features"]
    .to_list()
)

removed_features = {
    feature
    for row in removed_features
    for feature in row.split(" | ")
    if feature
}

selected_features = [
    feature
    for feature in feature_cols
    if feature not in removed_features
]

print(f"Step: {SELECTED_STEP}")
print(f"Features: {len(selected_features)}")

selected_features

Step: 28
Features: 98


['тип_датчика',
 'тег_инженерной_системы',
 'родитель',
 'day_of_week',
 'value_cat__Норма__sum_3d',
 'value_cat__Обнаружен газ__sum_3d',
 'value_cat__Обнаружен дым__sum_3d',
 'value_cat__Обесточен__sum_3d',
 'value_float__mean_3d',
 'value_cat__Дыма нет__sum_2d',
 'value_cat__Обнаружен дым__sum_2d',
 'value_cat__Неопределен__sum_2d',
 'value_cat__Отключено устройство__sum_2d',
 'value_float__mean_2d',
 'value_cat__Неисправен__sum_1d',
 'value_cat__Норма__sum_1d',
 'value_cat__Обнаружен дым__sum_1d',
 'value_cat__В норме от +3 до +40__sum_1d',
 'value_cat__Неопределен__sum_1d',
 'value_cat__Отключено устройство__sum_1d',
 'value_cat__Неисправен__sum_12h',
 'value_cat__В норме от +3 до +40__sum_12h',
 'value_cat__Неисправен__sum_6h',
 'value_cat__Норма__sum_6h',
 'value_cat__Обнаружен газ__sum_6h',
 'value_cat__Обнаружен дым__sum_6h',
 'value_cat__Неопределен__sum_6h',
 'value_cat__Неисправен__sum_4h',
 'value_cat__Обнаружен дым__sum_4h',
 'value_cat__Неисправен__sum_2h',
 'value_cat__О

In [17]:
# %%
def get_n_rows(
    split: str,
    chunks: list[int],
) -> int:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .select(pl.len())
        .collect()
        .item()
    )


n_train = get_n_rows(
    "train",
    TRAIN_CHUNKS,
)

n_val = get_n_rows(
    "val",
    VAL_CHUNKS,
)

n_test = get_n_rows(
    "test",
    TEST_CHUNKS,
)


print(f"Train: {n_train:,}")
print(f"Val:   {n_val:,}")
print(f"Test:  {n_test:,}")

Train: 5,118,137
Val:   159,908
Test:  2,318,098
